# 41 - Preparar dados para modelagem e comparar normalizacoes

Nesta branch experimental, a base de entrada e `notas-pessoais` (notas fiscais pessoais do autor, convertidas pelo notebook 22). Essa base nao possui NCM; as colunas `codigo_ncm_mais_frequente` e `ncm_tipo_mais_frequente` ficam vazias, mas sao mantidas porque fazem parte do schema esperado pelos notebooks seguintes. Como ha menos de mil descricoes unicas, a amostragem fica desabilitada e a base completa e salva em `data/processed/notas-pessoais_semstopwords/`.

Este notebook repete o fluxo do notebook 40: carrega os Parquets intermediarios, seleciona descricoes, remove nulos e vazios, normaliza texto, consolida duplicatas preservando frequencia, agrega metadados e valida o resultado.

Tambem cria colunas experimentais sem substituir `descricao_normalizada`: uma padroniza abreviacoes e unidades; outra uniformiza a separacao entre numeros e unidades; e uma terceira mascara valores numericos. As colunas experimentais sao alternativas de entrada para os proximos experimentos de TF-IDF e podem conter descricoes repetidas, por isso nao alteram a chave de consolidacao nem a frequencia. A gravacao final usa um arquivo separado e falha se ele ja existir.

Opcionalmente, remove uma lista conservadora de stop words das quatro representações antes da consolidação. A nova chave pode unir descrições antes distintas; frequências e metadados são recalculados a partir dos itens restantes. Descrições normalizadas que ficam vazias são excluídas e contabilizadas. Não há lematização.

Também permite gerar uma base amostral com tamanho máximo configurável e seed 42. A seleção ocorre após a preparação e a validação da base completa, usando amostragem uniforme sem reposição, como nos notebooks da série 70.

In [1]:
from __future__ import annotations

import sys
import re
import unicodedata
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    source_dir = path / "src"
    if source_dir.exists():
        sys.path.append(str(source_dir))
        break
else:
    raise FileNotFoundError("Nao encontrei a pasta src nos diretorios pais.")

from nofis_classifier.paths import find_project_root

## Configuracao

Defina `MANTER_DUPLICATAS = True` para gerar uma linha por item valido, sem consolidar descricoes normalizadas. Com `False`, o fluxo consolida descricoes e registra a frequencia, como antes. Cada modo usa um nome de saida proprio.

As regras experimentais abaixo sao deliberadamente conservadoras. Revise os exemplos e compare cada coluna em experimentos separados antes de escolher uma representacao para o modelo.

Defina `REMOVER_STOP_WORDS = True` para gerar a base **`2025_NCMs_filtrados_semstopwords`** (modo consolidado). Com `MANTER_DUPLICATAS = True`, o nome recebe `-com-duplicatas`. Com a remoção desabilitada, o modo consolidado mantém o nome convencional; o modo com duplicatas usa `-com-duplicatas`.

`STOP_WORDS` é uma lista local e editável, com artigos e conectivos. Por padrão, preserva `com`, `sem`, `nao` e `para`, assim como marcas, números e unidades que não constem da lista. A remoção considera tokens completos após a normalização; não remove trechos dentro de palavras. Ao editar a lista, use uma palavra por entrada. As configurações permanecem neste notebook, sem downloads ou novas dependências.

Defina `GERAR_AMOSTRA = True` para salvar somente uma amostra da base preparada. `MAX_DESCRIPTIONS = 10_000` é o limite inicial e `SAMPLE_RANDOM_STATE = 42` controla a reprodução. Quando houver menos linhas disponíveis, todas serão mantidas. A opção começa desabilitada para preservar a geração da base completa.

Com duplicatas consolidadas, a unidade amostral é a descrição normalizada única, sem ponderação por `frequencia`, como na série 70. Com duplicatas mantidas, a unidade amostral é a linha de item; descrições podem se repetir. A amostra não é estratificada por mês ou NCM. Todas as colunas e frequências das linhas selecionadas são preservadas, sem recontagem ou extrapolação para o dataset completo.

A seleção usa a base ordenada por `descricao_normalizada`; o resultado também é ordenado por essa coluna. O nome recebe `-amostra10000-seed42` com os parâmetros iniciais, em uma pasta própria. O número no nome é o limite solicitado; o resumo mostra o tamanho efetivo.

In [2]:
DATASET_NAME = "notas-pessoais"
MANTER_DUPLICATAS = False
REMOVER_STOP_WORDS = True
STOP_WORDS = {
    "a", "as", "o", "os", "de", "da", "das", "do", "dos", "e", "em",
    "no", "na", "nos", "nas", "ao", "aos", "um", "uma", "uns", "umas",
}
STOP_WORDS_EXAMPLE_COUNT = 20

GERAR_AMOSTRA = False  # A base de notas pessoais tem menos de mil descricoes unicas.
MAX_DESCRIPTIONS = 10_000
SAMPLE_RANDOM_STATE = 42

DESCRIPTION_COLUMNS = [
    "descricao_normalizada",
    "descricao_abreviacoes_unidades",
    "descricao_numero_unidade_padronizada",
    "descricao_numeros_mascarados",
]

if REMOVER_STOP_WORDS:
    EXPERIMENT_DATASET_NAME = f"{DATASET_NAME}_semstopwords"
    if MANTER_DUPLICATAS:
        EXPERIMENT_DATASET_NAME += "-com-duplicatas"
else:
    duplicate_suffix = "com-duplicatas" if MANTER_DUPLICATAS else "sem-duplicatas"
    EXPERIMENT_DATASET_NAME = f"{DATASET_NAME}-{duplicate_suffix}"
if GERAR_AMOSTRA:
    if isinstance(MAX_DESCRIPTIONS, bool) or not isinstance(MAX_DESCRIPTIONS, int) or MAX_DESCRIPTIONS < 1:
        raise ValueError("MAX_DESCRIPTIONS deve ser um inteiro positivo quando GERAR_AMOSTRA=True.")
    EXPERIMENT_DATASET_NAME += f"-amostra{MAX_DESCRIPTIONS}-seed{SAMPLE_RANDOM_STATE}"

FREQUENCY_COLUMN = "frequencia"
NUMERIC_SEQUENCE_PATTERN = r"\d{6,}" # remover sequencias de dígitos das descrições

PROJECT_ROOT = find_project_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / DATASET_NAME
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / EXPERIMENT_DATASET_NAME

# Mapeamentos testados na coluna descricao_abreviacoes_unidades.
ABBREVIATION_MAP = {
    "und": "un",
    "unidade": "un",
    "unidades": "un",
    "quilo": "kg",
    "quilos": "kg",
    "quilograma": "kg",
    "quilogramas": "kg",
    "grama": "g",
    "gramas": "g",
    "litro": "l",
    "litros": "l",
    "lt": "l",
    "mililitro": "ml",
    "mililitros": "ml",
    "pct": "pacote",
    "c": "com",
    "s": "sem",
}

INTERIM_DIR, PROCESSED_DIR

(WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/interim/notas-pessoais'),
 WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/processed/notas-pessoais_semstopwords'))

## Funcoes de preparacao

In [3]:
def normalize_column_name(column_name: str) -> str:
    """Normalize a column name enough to detect expected fields."""
    text = unicodedata.normalize("NFKD", str(column_name))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def find_column(columns: pd.Index, required_tokens: tuple[str, ...]) -> str:
    """Find the first column whose normalized name contains all required tokens."""
    for column in columns:
        normalized = normalize_column_name(column)
        if all(token in normalized for token in required_tokens):
            return column
    raise KeyError(f"Nao encontrei coluna com tokens {required_tokens}. Colunas: {list(columns)}")


def remove_long_numeric_sequences(value: object) -> str:
    """Remove digit sequences with six or more digits from a description."""
    if pd.isna(value):
        return ""

    text = re.sub(NUMERIC_SEQUENCE_PATTERN, " ", str(value))
    return re.sub(r"\s+", " ", text).strip()


def normalize_description(value: object) -> str:
    """Normalize product descriptions for TF-IDF and duplicate consolidation."""
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_month(period: str) -> str:
    """Convert YYYY-MM to YYYYMM for artifact names."""
    return period.replace("-", "")


def most_frequent_by_description(
    frame: pd.DataFrame,
    value_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Return the most common value per normalized description."""
    valid = frame[["descricao_normalizada", value_column]].dropna(subset=[value_column]).copy()
    valid = valid[valid[value_column].astype(str).str.strip() != ""]
    if valid.empty:
        return pd.DataFrame(columns=["descricao_normalizada", output_column])

    weighted = (
        valid.groupby(["descricao_normalizada", value_column], as_index=False)
        .size()
        .sort_values(["descricao_normalizada", "size"], ascending=[True, False])
    )
    return weighted.drop_duplicates("descricao_normalizada")[["descricao_normalizada", value_column]].rename(
        columns={value_column: output_column}
    )

In [4]:
def normalize_abbreviations_and_units(value: object) -> str:
    """Standardize a small, explicit set of abbreviations and unit names."""
    if pd.isna(value):
        return ""

    # Expand slash abbreviations before punctuation normalization removes `/`.
    raw_text = re.sub(r"(?i)\bc\s*/", "com ", str(value))
    raw_text = re.sub(r"(?i)\bs\s*/", "sem ", raw_text)
    text = normalize_description(raw_text)
    for source, replacement in ABBREVIATION_MAP.items():
        text = re.sub(rf"\b{re.escape(source)}\b", replacement, text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_number_unit_spacing(value: object) -> str:
    """Make number/unit spacing consistent while retaining numeric values."""
    text = normalize_description(value)
    # `500ml`, `500 ml` -> `500 ml`; same for common units.
    text = re.sub(r"(?<=\d)(kg|mg|g|ml|l|un)\b", r" \1", text)
    return re.sub(r"\s+", " ", text).strip()


def mask_numeric_values(value: object) -> str:
    """Replace numeric values with NUMERO while retaining unit tokens."""
    text = normalize_number_unit_spacing(value)
    text = re.sub(r"\b\d+(?:\s+\d+)*\b", " numero ", text)
    return re.sub(r"\s+", " ", text).strip()



def remove_stop_words(value: object, stop_words: set[str]) -> str:
    """Remove whole tokens from an already normalized description."""
    if pd.isna(value):
        return ""
    return " ".join(token for token in str(value).split() if token not in stop_words)


def sample_modeling_items(
    frame: pd.DataFrame, max_descriptions: int, random_state: int
) -> pd.DataFrame:
    """Sample prepared rows uniformly, preserving metadata and original frequencies."""
    if isinstance(max_descriptions, bool) or not isinstance(max_descriptions, int) or max_descriptions < 1:
        raise ValueError("O tamanho máximo da amostra deve ser um inteiro positivo.")
    ordered = frame.sort_values("descricao_normalizada", kind="stable").reset_index(drop=True)
    sampled = ordered.sample(n=min(max_descriptions, len(ordered)), random_state=random_state, replace=False)
    return sampled.sort_values("descricao_normalizada", kind="stable").reset_index(drop=True)

## Testes rapidos da normalizacao

In [5]:
normalization_examples = {
    " ARROZ TIPO 1 5KG ": "arroz tipo 1 5kg",
    "CAFE TORRADO 500g": "cafe torrado 500g",
    "LEITE UHT 1L": "leite uht 1l",
    "ACUCAR-CRISTAL 2 KG": "acucar cristal 2 kg",
}

for raw, expected in normalization_examples.items():
    normalized = normalize_description(raw)
    assert normalized == expected, (raw, normalized, expected)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

informative_examples = ["QUEIJO MUCARELA, KG", "Couve Flor (Hortifruti) Ceasa", "PIMENTAO VERMELHA CEASA"]
for raw in informative_examples:
    normalized = normalize_description(raw)
    assert len(normalized) >= 3, (raw, normalized)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

duplicate_check = pd.DataFrame(
    {
        "descricao_original": ["Arroz Tipo 1", " arroz tipo 1 ", "ARROZ TIPO 1"],
    }
)
duplicate_check["descricao_normalizada"] = duplicate_check["descricao_original"].map(normalize_description)
duplicate_frequency = duplicate_check.groupby("descricao_normalizada").size().iloc[0]
assert duplicate_frequency == len(duplicate_check)

numeric_sequence_examples = {
    "ARROZ COD123456789 TIPO 1": "ARROZ COD TIPO 1",
    "LEITE 12345 1L": "LEITE 12345 1L",
    "ITEM 123456 E LOTE 987654321": "ITEM E LOTE",
}
for raw, expected in numeric_sequence_examples.items():
    actual = remove_long_numeric_sequences(raw)
    assert actual == expected, (raw, actual, expected)

sequence_check = pd.DataFrame({"descricao_original": ["ARROZ COD123456 TIPO 1"]})
sequence_check["descricao_original"] = sequence_check["descricao_original"].map(
    remove_long_numeric_sequences
)
assert len(sequence_check) == 1
assert sequence_check.loc[0, "descricao_original"] == "ARROZ COD TIPO 1"


# Verificações independentes da opção habilitada para a execução principal.
example_stop_words = {"de", "da", "e"}  # Fixture fixa; a lista configurável pode ser editada.
assert {normalize_description(word) for word in STOP_WORDS}.isdisjoint({"com", "sem", "nao", "para", "kg", "g", "l", "ml", "un"})
stopword_examples_check = {
    "CAFE DE CEVADA": "cafe cevada",
    "LEITE SEM LACTOSE COM CALCIO": "leite sem lactose com calcio",
    "NAO CONTEM ACUCAR": "nao contem acucar",
    "CERA PARA MADEIRA 500 ML": "cera para madeira 500 ml",
    "DETERGENTE OMO 1 L": "detergente omo 1 l",
    "DE DA E": "",
    "": "",
}
for raw, expected in stopword_examples_check.items():
    normalized = normalize_description(raw)
    actual = remove_stop_words(normalized, example_stop_words)
    assert actual == expected, (raw, actual, expected)
assert remove_stop_words(None, example_stop_words) == ""
assert remove_stop_words("cafe de cevada", set()) == "cafe de cevada"
assert remove_stop_words("cafe de cevada", example_stop_words) == remove_stop_words(
    "cafe cevada", example_stop_words
)

# A amostragem mantém as frequências e reproduz a seleção da série 70.
sampling_check = pd.DataFrame({
    "descricao_normalizada": ["leite", "arroz", "cafe", "feijao", "banana"],
    "frequencia": [100, 1, 3, 2, 5],
    "primeiro_mes": ["2025-01"] * 5,
})
sampling_result = sample_modeling_items(sampling_check, 3, 42)
sampling_expected = (
    sampling_check.sort_values("descricao_normalizada").reset_index(drop=True)
    .sample(n=3, random_state=42)
    .sort_values("descricao_normalizada").reset_index(drop=True)
)
pd.testing.assert_frame_equal(sampling_result, sampling_expected)
pd.testing.assert_frame_equal(sampling_result, sample_modeling_items(sampling_check, 3, 42))
assert sampling_result["descricao_normalizada"].is_unique
assert len(sample_modeling_items(sampling_check, 10_000, 42)) == len(sampling_check)

## Variantes experimentais de texto

`descricao_normalizada` permanece como baseline. As variantes abaixo usam regras deterministicas e deixam quantidades disponiveis em colunas alternativas: abreviacoes/unidades padronizadas, espacamento numero-unidade canonico e numeros substituidos por `numero`. `c` e `s` sao expandidos apenas como tokens isolados, portanto inspecione exemplos reais antes de interpretar o impacto.

Quando `REMOVER_STOP_WORDS` está habilitada, a mesma lista é aplicada a todas essas representações após sua geração, antes de agrupar os itens por `descricao_normalizada`.

In [6]:
variant_examples = {
    "ARROZ 1KG": ("arroz 1kg", "arroz 1 kg", "arroz numero kg"),
    "ARROZ 1 KG": ("arroz 1 kg", "arroz 1 kg", "arroz numero kg"),
    "LEITE 500ML": ("leite 500ml", "leite 500 ml", "leite numero ml"),
    "LEITE 500 ML": ("leite 500 ml", "leite 500 ml", "leite numero ml"),
    "CAFE 250 GRAMAS": ("cafe 250 gramas", "cafe 250 gramas", "cafe numero gramas"),
}
for raw, expected in variant_examples.items():
    actual = (
        normalize_description(raw),
        normalize_number_unit_spacing(raw),
        mask_numeric_values(raw),
    )
    assert actual == expected, (raw, actual, expected)

assert normalize_abbreviations_and_units("LEITE 1 LITRO") == "leite 1 l"
assert normalize_abbreviations_and_units("SABAO 2 UND") == "sabao 2 un"
assert normalize_abbreviations_and_units("CAFE C/ ACUCAR") == "cafe com acucar"



## Carregar Parquets intermediarios

In [7]:
parquet_files = sorted(INTERIM_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {INTERIM_DIR}")

frames = []
for parquet_file in parquet_files:
    frame = pd.read_parquet(parquet_file)
    frame["competencia"] = parquet_file.stem
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Arquivos carregados: {len(parquet_files)}")
print(f"Linhas intermediarias: {len(df):,}")
df.head()

Arquivos carregados: 13
Linhas intermediarias: 1,931


,CHAVE DE ACESSO,MODELO,SÉRIE,NÚMERO,NATUREZA DA OPERAÇÃO,DATA EMISSÃO,CPF/CNPJ Emitente,RAZÃO SOCIAL EMITENTE,INSCRIÇÃO ESTADUAL EMITENTE,UF EMITENTE,...,DESCRIÇÃO DO PRODUTO/SERVIÇO,CÓDIGO NCM/SH,NCM/SH (TIPO DE PRODUTO),CFOP,QUANTIDADE,UNIDADE,VALOR UNITÁRIO,VALOR TOTAL,arquivo_origem,competencia
0,43251107718633004176650180000298181939351260,65,18,29818,NaN,09/11/2025 13:18:18,NaN,NaN,NaN,NaN,...,ABACAXI PEROLA UN,NaN,NaN,NaN,"2,000",UN,"9,9800","19,96",notas-lote-1.csv,2025-11
1,43251107718633004176650180000298181939351260,65,18,29818,NaN,09/11/2025 13:18:18,NaN,NaN,NaN,NaN,...,BANANA PRATA kg,NaN,NaN,NaN,"0,715",KG,"7,9900","5,71",notas-lote-1.csv,2025-11
2,43251107718633004176650180000298181939351260,65,18,29818,NaN,09/11/2025 13:18:18,NaN,NaN,NaN,NaN,...,PESSEGO NACIONAL kg,NaN,NaN,NaN,"0,910",KG,"12,4900","11,37",notas-lote-1.csv,2025-11
3,43251107718633004176650180000298181939351260,65,18,29818,NaN,09/11/2025 13:18:18,NaN,NaN,NaN,NaN,...,AMEIXA IMPORT kg,NaN,NaN,NaN,"0,280",KG,"24,9000","6,97",notas-lote-1.csv,2025-11
4,43251107718633004176650180000298181939351260,65,18,29818,NaN,09/11/2025 13:18:18,NaN,NaN,NaN,NaN,...,MACA GALA kg,NaN,NaN,NaN,"0,685",KG,"13,9800","9,58",notas-lote-1.csv,2025-11


## Selecionar descricoes

In [8]:
description_col = find_column(df.columns, ("DESCRI", "PRODUTO"))

support_columns = {
    "codigo_ncm": None,
    "ncm_tipo": None,
}

try:
    support_columns["codigo_ncm"] = find_column(df.columns, ("NCM", "SH"))
except KeyError:
    pass

try:
    support_columns["ncm_tipo"] = find_column(df.columns, ("TIPO", "PRODUTO"))
except KeyError:
    pass

selected_columns = [description_col, "competencia"]
selected_columns.extend(column for column in support_columns.values() if column is not None)

rename_columns = {description_col: "descricao_original"}
if support_columns["codigo_ncm"] is not None:
    rename_columns[support_columns["codigo_ncm"]] = "codigo_ncm"
if support_columns["ncm_tipo"] is not None:
    rename_columns[support_columns["ncm_tipo"]] = "ncm_tipo"

df_selected = df[selected_columns].rename(columns=rename_columns)

# Bases sem NCM (como notas-pessoais) trazem as colunas, mas totalmente vazias.
print({key: (column, int(df[column].notna().sum()) if column else None) for key, column in support_columns.items()})
df_selected.head()

{'codigo_ncm': ('CÓDIGO NCM/SH', 0), 'ncm_tipo': ('NCM/SH (TIPO DE PRODUTO)', 0)}


,descricao_original,competencia,codigo_ncm,ncm_tipo
0,ABACAXI PEROLA UN,2025-11,NaN,NaN
1,BANANA PRATA kg,2025-11,NaN,NaN
2,PESSEGO NACIONAL kg,2025-11,NaN,NaN
3,AMEIXA IMPORT kg,2025-11,NaN,NaN
4,MACA GALA kg,2025-11,NaN,NaN


## Remover sequencias numericas longas, normalizar e consolidar duplicatas

Cadeias com seis ou mais digitos consecutivos sao removidas do texto da descricao antes da normalizacao. A linha e preservada quando restam outros termos; descricoes que ficam vazias continuam sujeitas ao filtro existente de descricoes invalidas.

In [9]:
input_rows = len(df_selected)

df_valid = df_selected.dropna(subset=["descricao_original"]).copy()
df_valid["descricao_original"] = df_valid["descricao_original"].astype(str).str.strip()
df_valid = df_valid[df_valid["descricao_original"] != ""].copy()

numeric_sequences_removed = int(
    df_valid["descricao_original"].str.count(NUMERIC_SEQUENCE_PATTERN).sum()
)
df_valid["descricao_original"] = df_valid["descricao_original"].map(
    remove_long_numeric_sequences
)

df_valid["descricao_normalizada"] = df_valid["descricao_original"].map(normalize_description)
df_valid = df_valid[df_valid["descricao_normalizada"] != ""].copy()

# Variantes para comparar em experimentos; a consolidacao segue o baseline.
df_valid["descricao_abreviacoes_unidades"] = df_valid["descricao_original"].map(
    normalize_abbreviations_and_units
)
df_valid["descricao_numero_unidade_padronizada"] = df_valid["descricao_original"].map(
    normalize_number_unit_spacing
)
df_valid["descricao_numeros_mascarados"] = df_valid["descricao_original"].map(mask_numeric_values)

valid_rows_before_stopwords = len(df_valid)

print(f"Linhas com coluna de descricao: {input_rows:,}")
print(f"Linhas com descricao valida antes de stop words: {valid_rows_before_stopwords:,}")
print(f"Sequencias numericas removidas: {numeric_sequences_removed:,}")
print(f"Linhas removidas por nulos/vazios: {input_rows - valid_rows_before_stopwords:,}")

Linhas com coluna de descricao: 1,931
Linhas com descricao valida antes de stop words: 1,931
Sequencias numericas removidas: 9
Linhas removidas por nulos/vazios: 0


## Remoção opcional de stop words e comparação antes/depois

A remoção acontece antes de consolidar duplicatas e não altera `descricao_original`. Por exemplo, `cafe de cevada` e `cafe cevada` passam a compartilhar uma chave; suas ocorrências e competências são agregadas juntas. Um item só é excluído se a representação principal ficar vazia. Variantes alternativas vazias são contabilizadas na tabela para inspeção.

As tabelas mostram alterações nas quatro representações e exemplos distintos da representação principal. O total de itens excluídos por stop words é separado das exclusões anteriores por nulos ou normalização.


In [10]:
unique_descriptions_before_stopwords = df_valid["descricao_normalizada"].nunique()
stopword_changed_items = pd.Series(False, index=df_valid.index)
stopword_empty_rows = 0
stopword_summary_rows = []
stopword_examples = pd.DataFrame(columns=["descricao_original", "antes", "depois"])

if REMOVER_STOP_WORDS:
    normalized_stop_words = {normalize_description(word) for word in STOP_WORDS}
    if any(not word or len(word.split()) != 1 for word in normalized_stop_words):
        raise ValueError("Cada entrada de STOP_WORDS deve conter exatamente um token após normalizar.")

    for column in DESCRIPTION_COLUMNS:
        before = df_valid[column].copy()
        after = before.map(lambda text: remove_stop_words(text, normalized_stop_words))
        changed = before.ne(after)
        stopword_changed_items |= changed
        stopword_summary_rows.append({
            "representacao": column,
            "itens_alterados": int(changed.sum()),
            "tokens_removidos": int((before.str.split().str.len() - after.str.split().str.len()).sum()),
            "itens_com_texto_vazio": int(after.eq("").sum()),
        })
        df_valid[column] = after
        if column == "descricao_normalizada":
            stopword_examples = (
                pd.DataFrame({"descricao_original": df_valid["descricao_original"], "antes": before, "depois": after})
                .loc[changed].drop_duplicates("antes").head(STOP_WORDS_EXAMPLE_COUNT)
            )

    stopword_empty_rows = int(df_valid["descricao_normalizada"].eq("").sum())
    df_valid = df_valid.loc[df_valid["descricao_normalizada"].ne("")].copy()

valid_rows = len(df_valid)
unique_descriptions_after_stopwords = df_valid["descricao_normalizada"].nunique()
print(f"Remoção de stop words habilitada: {REMOVER_STOP_WORDS}")
print(f"Itens alterados em pelo menos uma representação: {int(stopword_changed_items.sum()):,}")
print(f"Itens excluídos por descrição vazia após stop words: {stopword_empty_rows:,}")
print(f"Itens restantes: {valid_rows:,}")
print(f"Descrições únicas antes/depois: {unique_descriptions_before_stopwords:,} / {unique_descriptions_after_stopwords:,}")
if REMOVER_STOP_WORDS:
    display(pd.DataFrame(stopword_summary_rows))
    display(stopword_examples.reset_index(drop=True))


Remoção de stop words habilitada: True
Itens alterados em pelo menos uma representação: 104
Itens excluídos por descrição vazia após stop words: 0
Itens restantes: 1,931
Descrições únicas antes/depois: 988 / 987


,representacao,itens_alterados,tokens_removidos,itens_com_texto_vazio
0,descricao_normalizada,104,120,0
1,descricao_abreviacoes_unidades,104,120,0
2,descricao_numero_unidade_padronizada,104,120,0
3,descricao_numeros_mascarados,104,120,0


,descricao_original,antes,depois
0,DES AER NIVEA M INV BLACK E WHITE 200ml,des aer nivea m inv black e white 200ml,des aer nivea m inv black white 200ml
1,ERVILHA E MILHO ODERICH LT 200g,ervilha e milho oderich lt 200g,ervilha milho oderich lt 200g
2,COSTELA EM TIRAS (VACUO) kg,costela em tiras vacuo kg,costela tiras vacuo kg
3,DIVINE TRUFA DE CHOC.CREME SU,divine trufa de choc creme su,divine trufa choc creme su
4,QUINTA DO MORGADO ESP MOSC,quinta do morgado esp mosc,quinta morgado esp mosc
5,REV - PAO DE QUEIJO TRADICIONAL 400G,rev pao de queijo tradicional 400g,rev pao queijo tradicional 400g
6,BEST CUP 750ML DOCE DE LEITE,best cup 750ml doce de leite,best cup 750ml doce leite
7,PICOLE BOMBOM NO PALITO BRIGADEIRO,picole bombom no palito brigadeiro,picole bombom palito brigadeiro
8,"AGUA MIN FONTE DA LOMBA S/GAS 1,5L",agua min fonte da lomba s gas 1 5l,agua min fonte lomba s gas 1 5l
9,BOMBOM ARCOR BON O BON BEIJIN 15g,bombom arcor bon o bon beijin 15g,bombom arcor bon bon beijin 15g


In [11]:
if MANTER_DUPLICATAS:
    df_modeling = df_valid.copy()
    df_modeling[FREQUENCY_COLUMN] = 1
    df_modeling["descricao_original_exemplo"] = df_modeling["descricao_original"]
    df_modeling["descricoes_originais_distintas"] = 1
    df_modeling["primeiro_mes"] = df_modeling["competencia"]
    df_modeling["ultimo_mes"] = df_modeling["competencia"]
    df_modeling["meses_presentes"] = 1
    rename_columns = {
        "codigo_ncm": "codigo_ncm_mais_frequente",
        "ncm_tipo": "ncm_tipo_mais_frequente",
    }
    df_modeling = df_modeling.rename(
        columns={key: value for key, value in rename_columns.items() if key in df_modeling.columns}
    )
    df_modeling = df_modeling.sort_values(
        ["competencia", "descricao_normalizada"], ascending=[True, True]
    ).reset_index(drop=True)
else:
    aggregations = {
        FREQUENCY_COLUMN: ("descricao_original", "size"),
        "descricao_original_exemplo": ("descricao_original", "first"),
        "descricoes_originais_distintas": ("descricao_original", "nunique"),
        "primeiro_mes": ("competencia", "min"),
        "ultimo_mes": ("competencia", "max"),
        "meses_presentes": ("competencia", "nunique"),
        "descricao_abreviacoes_unidades": ("descricao_abreviacoes_unidades", "first"),
        "descricao_numero_unidade_padronizada": ("descricao_numero_unidade_padronizada", "first"),
        "descricao_numeros_mascarados": ("descricao_numeros_mascarados", "first"),
    }

    df_modeling = (
        df_valid.groupby("descricao_normalizada", as_index=False)
        .agg(**aggregations)
        .sort_values([FREQUENCY_COLUMN, "descricao_normalizada"], ascending=[False, True])
        .reset_index(drop=True)
    )

    if "codigo_ncm" in df_valid.columns:
        ncm_mode = most_frequent_by_description(df_valid, "codigo_ncm", "codigo_ncm_mais_frequente")
        df_modeling = df_modeling.merge(ncm_mode, on="descricao_normalizada", how="left")

    if "ncm_tipo" in df_valid.columns:
        ncm_tipo_mode = most_frequent_by_description(df_valid, "ncm_tipo", "ncm_tipo_mais_frequente")
        df_modeling = df_modeling.merge(ncm_tipo_mode, on="descricao_normalizada", how="left")

df_modeling.head(10)


,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,carne moida bov 2 agulha kg,37,CARNE MOIDA BOV 2 AGULHA kg,1,2025-12,2026-10,8,carne moida bov 2 agulha kg,carne moida bov 2 agulha kg,carne moida bov numero agulha kg,NaN,NaN
1,beb frukito fr citr pet 2l,30,BEB FRUKITO FR CITR PET 2L,1,2025-11,2026-10,11,beb frukito fr citr pet 2l,beb frukito fr citr pet 2 l,beb frukito fr citr pet numero l,NaN,NaN
2,tomate longa vida kg,24,TOMATE LONGA VIDA kg,2,2025-11,2026-10,12,tomate longa vida kg,tomate longa vida kg,tomate longa vida kg,NaN,NaN
3,ovos bco c 20 un,23,OVOS BCO C/20 UN,1,2025-11,2026-10,12,ovos bco com 20 un,ovos bco c 20 un,ovos bco c numero un,NaN,NaN
4,refri sprite s acuc pet 2l,23,REFRI SPRITE S/ACUC PET 2L,1,2025-11,2026-10,11,refri sprite sem acuc pet 2l,refri sprite s acuc pet 2 l,refri sprite s acuc pet numero l,NaN,NaN
5,manga tommy kg,20,MANGA TOMMY kg,1,2026-02,2026-10,9,manga tommy kg,manga tommy kg,manga tommy kg,NaN,NaN
6,cebola nacional kg,18,CEBOLA NACIONAL kg,1,2025-11,2026-10,8,cebola nacional kg,cebola nacional kg,cebola nacional kg,NaN,NaN
7,abacaxi perola un,16,ABACAXI PEROLA UN,1,2025-11,2026-10,9,abacaxi perola un,abacaxi perola un,abacaxi perola un,NaN,NaN
8,envelope figurinha copa 2026,15,ENVELOPE FIGURINHA COPA 2026,1,2026-05,2026-05,1,envelope figurinha copa 2026,envelope figurinha copa 2026,envelope figurinha copa numero,NaN,NaN
9,batata bca kg,14,BATATA BCA kg,1,2025-11,2026-10,9,batata bca kg,batata bca kg,batata bca kg,NaN,NaN


## Validacoes finais

In [12]:
assert valid_rows_before_stopwords == valid_rows + stopword_empty_rows
assert not df_modeling.empty
assert df_modeling["descricao_normalizada"].notna().all()
assert (df_modeling["descricao_normalizada"].str.strip() != "").all()
assert int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows
if not MANTER_DUPLICATAS:
    assert df_modeling["descricao_normalizada"].is_unique
else:
    assert len(df_modeling) == valid_rows
    assert df_modeling[FREQUENCY_COLUMN].eq(1).all()

print(f"Linhas na base: {len(df_modeling):,}")
print(f"Descricoes unicas normalizadas: {df_modeling['descricao_normalizada'].nunique():,}")
print(f"Duplicatas mantidas: {MANTER_DUPLICATAS}")
print(f"Linhas validas consolidadas: {int(df_modeling[FREQUENCY_COLUMN].sum()):,}")
print(f"Frequencia preservada: {int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows}")

Linhas na base: 987
Descricoes unicas normalizadas: 987
Duplicatas mantidas: False
Linhas validas consolidadas: 1,931
Frequencia preservada: True


## Amostragem aleatória opcional da base preparada

A base completa já foi validada acima. Nesta etapa, selecionamos no máximo `MAX_DESCRIPTIONS` linhas sem reposição, com `SAMPLE_RANDOM_STATE`. No modo consolidado, a seleção é uniforme por descrição única; frequências maiores não aumentam a chance de seleção.

`frequencia` e os metadados continuam descrevendo as ocorrências originais das descrições selecionadas. Por isso, a soma de frequências da amostra pode ser maior que seu número de linhas e menor que o total da base completa. No modo com duplicatas, cada linha selecionada continua com frequência 1. Não há divisão de treino/validação nesta etapa.


In [13]:
modeling_rows_before_sampling = len(df_modeling)
frequency_before_sampling = int(df_modeling[FREQUENCY_COLUMN].sum())

if GERAR_AMOSTRA:
    df_modeling = sample_modeling_items(df_modeling, MAX_DESCRIPTIONS, SAMPLE_RANDOM_STATE)
    assert len(df_modeling) == min(MAX_DESCRIPTIONS, modeling_rows_before_sampling)
else:
    assert len(df_modeling) == modeling_rows_before_sampling

assert not df_modeling.empty
assert int(df_modeling[FREQUENCY_COLUMN].sum()) <= frequency_before_sampling
if not MANTER_DUPLICATAS:
    assert df_modeling["descricao_normalizada"].is_unique
else:
    assert df_modeling[FREQUENCY_COLUMN].eq(1).all()

print(f"Amostragem habilitada: {GERAR_AMOSTRA}")
if GERAR_AMOSTRA:
    print(f"Limite solicitado: {MAX_DESCRIPTIONS:,}; seed: {SAMPLE_RANDOM_STATE}")
print(f"Linhas na base completa: {modeling_rows_before_sampling:,}")
print(f"Linhas selecionadas para saída: {len(df_modeling):,}")
print(f"Ocorrências representadas na saída: {int(df_modeling[FREQUENCY_COLUMN].sum()):,} de {frequency_before_sampling:,}")
print(f"Dataset de saída: {EXPERIMENT_DATASET_NAME}")
display(df_modeling.head())


Amostragem habilitada: False
Linhas na base completa: 987
Linhas selecionadas para saída: 987
Ocorrências representadas na saída: 1,931 de 1,931
Dataset de saída: notas-pessoais_semstopwords


,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,carne moida bov 2 agulha kg,37,CARNE MOIDA BOV 2 AGULHA kg,1,2025-12,2026-10,8,carne moida bov 2 agulha kg,carne moida bov 2 agulha kg,carne moida bov numero agulha kg,NaN,NaN
1,beb frukito fr citr pet 2l,30,BEB FRUKITO FR CITR PET 2L,1,2025-11,2026-10,11,beb frukito fr citr pet 2l,beb frukito fr citr pet 2 l,beb frukito fr citr pet numero l,NaN,NaN
2,tomate longa vida kg,24,TOMATE LONGA VIDA kg,2,2025-11,2026-10,12,tomate longa vida kg,tomate longa vida kg,tomate longa vida kg,NaN,NaN
3,ovos bco c 20 un,23,OVOS BCO C/20 UN,1,2025-11,2026-10,12,ovos bco com 20 un,ovos bco c 20 un,ovos bco c numero un,NaN,NaN
4,refri sprite s acuc pet 2l,23,REFRI SPRITE S/ACUC PET 2L,1,2025-11,2026-10,11,refri sprite sem acuc pet 2l,refri sprite s acuc pet 2 l,refri sprite s acuc pet numero l,NaN,NaN


## Salvar base processada

A saida e gravada em `data/processed/` na pasta definida por `EXPERIMENT_DATASET_NAME`, separada da base do notebook 40. Com duplicatas mantidas, cada linha representa um item valido e frequencia vale 1. No modo consolidado, as colunas alternativas representam a primeira descricao de cada grupo baseline.

Para o período anual, com remoção habilitada e duplicatas consolidadas, o Parquet será `data/processed/2025_NCMs_filtrados_semstopwords/2025_NCMs_filtrados_semstopwords-202501-202512.parquet`. O CSV auxiliar mantém o mesmo prefixo. Os dois destinos são verificados antes de iniciar a gravação; arquivos existentes nunca são sobrescritos. Se editar a lista para outro experimento, escolha um novo `EXPERIMENT_DATASET_NAME`.

Com `GERAR_AMOSTRA = True`, esta célula salva apenas as linhas selecionadas. Por exemplo, com stop words removidas, duplicatas consolidadas, limite 10.000 e seed 42, o dataset será `2025_NCMs_filtrados_semstopwords-amostra10000-seed42`. A base completa permanece disponível ao executar novamente com a opção desabilitada; a amostragem não grava uma segunda cópia completa automaticamente.

In [14]:
start_period = compact_month(min(file.stem for file in parquet_files))
end_period = compact_month(max(file.stem for file in parquet_files))

# Esta célula grava o Parquet e o CSV auxiliar. Verifica ambos antes de gravar.
output_stem = f"{EXPERIMENT_DATASET_NAME}-{start_period}-{end_period}"
output_path = PROCESSED_DIR / f"{output_stem}.parquet"
csv_output_path = PROCESSED_DIR / f"{output_stem}.csv"
existing_outputs = [path for path in (output_path, csv_output_path) if path.exists()]
if existing_outputs:
    raise FileExistsError(
        f"Arquivos processados já existem: {existing_outputs}. "
        "Escolha outro EXPERIMENT_DATASET_NAME para preservar as bases existentes."
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_modeling.to_parquet(output_path, index=False)
df_modeling.to_csv(csv_output_path, index=False)

output_path

WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/processed/notas-pessoais_semstopwords/notas-pessoais_semstopwords-202511-202611.parquet')

In [15]:
df_modeling.head()

,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,carne moida bov 2 agulha kg,37,CARNE MOIDA BOV 2 AGULHA kg,1,2025-12,2026-10,8,carne moida bov 2 agulha kg,carne moida bov 2 agulha kg,carne moida bov numero agulha kg,NaN,NaN
1,beb frukito fr citr pet 2l,30,BEB FRUKITO FR CITR PET 2L,1,2025-11,2026-10,11,beb frukito fr citr pet 2l,beb frukito fr citr pet 2 l,beb frukito fr citr pet numero l,NaN,NaN
2,tomate longa vida kg,24,TOMATE LONGA VIDA kg,2,2025-11,2026-10,12,tomate longa vida kg,tomate longa vida kg,tomate longa vida kg,NaN,NaN
3,ovos bco c 20 un,23,OVOS BCO C/20 UN,1,2025-11,2026-10,12,ovos bco com 20 un,ovos bco c 20 un,ovos bco c numero un,NaN,NaN
4,refri sprite s acuc pet 2l,23,REFRI SPRITE S/ACUC PET 2L,1,2025-11,2026-10,11,refri sprite sem acuc pet 2l,refri sprite s acuc pet 2 l,refri sprite s acuc pet numero l,NaN,NaN


## Analise breve do dataset final

In [16]:
analysis_summary = pd.Series(
    {
        "arquivos_origem": len(parquet_files),
        "linhas_originais": len(df),
        "linhas_validas_antes_stopwords": valid_rows_before_stopwords,
        "remover_stop_words": REMOVER_STOP_WORDS,
        "itens_alterados_stopwords": int(stopword_changed_items.sum()),
        "itens_excluidos_stopwords": stopword_empty_rows,
        "descricoes_unicas_antes_stopwords": unique_descriptions_before_stopwords,
        "descricoes_unicas_apos_stopwords": unique_descriptions_after_stopwords,
        "linhas_com_descricao_valida": valid_rows,
        "gerar_amostra": GERAR_AMOSTRA,
        "limite_amostra": MAX_DESCRIPTIONS if GERAR_AMOSTRA else None,
        "seed_amostra": SAMPLE_RANDOM_STATE if GERAR_AMOSTRA else None,
        "linhas_base_completa": modeling_rows_before_sampling,
        "linhas_base_saida": len(df_modeling),
        "descricoes_unicas_normalizadas": df_modeling["descricao_normalizada"].nunique(),
        "frequencia_base_completa": frequency_before_sampling,
        "frequencia_representada_saida": int(df_modeling[FREQUENCY_COLUMN].sum()),
        "frequencia_media_por_descricao": df_modeling[FREQUENCY_COLUMN].mean(),
        "frequencia_mediana_por_descricao": df_modeling[FREQUENCY_COLUMN].median(),
        "maior_frequencia": int(df_modeling[FREQUENCY_COLUMN].max()),
        "periodo_inicial": df_modeling["primeiro_mes"].min(),
        "periodo_final": df_modeling["ultimo_mes"].max(),
    }
)

analysis_summary

arquivos_origem                            13
linhas_originais                         1931
linhas_validas_antes_stopwords           1931
remover_stop_words                       True
itens_alterados_stopwords                 104
itens_excluidos_stopwords                   0
descricoes_unicas_antes_stopwords         988
descricoes_unicas_apos_stopwords          987
linhas_com_descricao_valida              1931
gerar_amostra                           False
limite_amostra                           None
seed_amostra                             None
linhas_base_completa                      987
linhas_base_saida                         987
descricoes_unicas_normalizadas            987
frequencia_base_completa                 1931
frequencia_representada_saida            1931
frequencia_media_por_descricao       1.956434
frequencia_mediana_por_descricao          1.0
maior_frequencia                           37
periodo_inicial                       2025-11
periodo_final                     

In [17]:
variant_collision_summary = pd.Series({
    "descricoes_baseline_unicas": df_modeling["descricao_normalizada"].nunique(),
    "unicas_abreviacoes_unidades": df_modeling["descricao_abreviacoes_unidades"].nunique(),
    "unicas_numero_unidade_padronizada": df_modeling["descricao_numero_unidade_padronizada"].nunique(),
    "unicas_numeros_mascarados": df_modeling["descricao_numeros_mascarados"].nunique(),
})
variant_collision_summary

descricoes_baseline_unicas           987
unicas_abreviacoes_unidades          987
unicas_numero_unidade_padronizada    987
unicas_numeros_mascarados            972
dtype: int64

In [18]:
frequency_bins = pd.cut(
    df_modeling[FREQUENCY_COLUMN],
    bins=[0, 1, 2, 5, 10, 50, 100, float("inf")],
    labels=["1", "2", "3-5", "6-10", "11-50", "51-100", "101+"],
)

frequency_profile = (
    frequency_bins.value_counts(sort=False)
    .rename_axis("faixa_frequencia")
    .reset_index(name="descricoes")
)
frequency_profile["percentual"] = (
    frequency_profile["descricoes"] / len(df_modeling) * 100
).round(2)

frequency_profile

,faixa_frequencia,descricoes,percentual
0,1,710,71.94
1,2,135,13.68
2,3-5,84,8.51
3,6-10,40,4.05
4,11-50,18,1.82
5,51-100,0,0.00
6,101+,0,0.00


In [19]:
top_descriptions = df_modeling[
    ["descricao_normalizada", "descricao_original_exemplo", FREQUENCY_COLUMN, "meses_presentes"]
].head(20)

top_descriptions

,descricao_normalizada,descricao_original_exemplo,frequencia,meses_presentes
0,carne moida bov 2 agulha kg,CARNE MOIDA BOV 2 AGULHA kg,37,8
1,beb frukito fr citr pet 2l,BEB FRUKITO FR CITR PET 2L,30,11
2,tomate longa vida kg,TOMATE LONGA VIDA kg,24,12
3,ovos bco c 20 un,OVOS BCO C/20 UN,23,12
4,refri sprite s acuc pet 2l,REFRI SPRITE S/ACUC PET 2L,23,11
5,manga tommy kg,MANGA TOMMY kg,20,9
6,cebola nacional kg,CEBOLA NACIONAL kg,18,8
7,abacaxi perola un,ABACAXI PEROLA UN,16,9
8,envelope figurinha copa 2026,ENVELOPE FIGURINHA COPA 2026,15,1
9,batata bca kg,BATATA BCA kg,14,9
